In [1]:
import pandas as pd
import datetime as dt
import os

df = pd.read_csv('C:/Users/Akansh/Desktop/ecommerce-project/cleaned_data.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
reference_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('InvoiceDate', 'nunique'),
    Monetary=('TotalPrice', 'sum')
)



rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1, 2, 3, 4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1, 2, 3, 4]).astype(int)



rfm['RFM_Score'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)



def segment_customer(row):
    if row['RFM_Score'] in ['444', '443', '434', '344']:
        return 'Champions'
    elif row['R_Score'] == 1:
        return 'At Risk / Lost'
    elif row['F_Score'] in [4, 3]:
        return 'Loyal Customers'
    else:
        return 'Others'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)

os.makedirs('data', exist_ok=True)
rfm.to_csv('data/rfm_segments.csv')
rfm.head()
        

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
CustomerID,,,,,,,,
12346,326,1,77183.60,1,1,4,114,At Risk / Lost
12347,2,7,4310.00,4,4,4,444,Champions
12348,75,4,1797.24,2,3,4,234,Loyal Customers
12349,19,1,1757.55,3,1,4,314,Others
12350,310,1,334.40,1,1,2,112,At Risk / Lost
